In [9]:
!pip install psycopg2

'pip' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [ ]:
import psycopg2 as pg
from datetime import datetime

In [ ]:
conexao = pg.connect(
    host="localhost",
    dbname="centro esportivo",
    user="postgres",
    password="postgres"
)
cursor = conexao.cursor()
print("conectado!")

conectado!


In [ ]:
class Cliente():
    id_cliente:int
    nome: str
    telefone: str 
    email: str

def __init__(self, nome: str, telefone: str | None, email: str | None, id_cliente: int | None = None):
            self.id_cliente = id_cliente
            self.nome = nome
            self.telefone = telefone
            self.email = email

def __str__(self):
            return f"Cliente(id_cliente={self.id_cliente}, nome={self.nome}, telefone={self.telefone}, email={self.email})"

def __repr__(self):
            return f"Cliente({self.id_cliente} - {self.nome})"

def criar_cliente(nome: str, telefone: str | None, email: str | None) -> Cliente | None:
        try:
            cursor.execute(
                "INSERT INTO cliente (nome, telefone, email) "
                "VALUES (%s, %s, %s) RETURNING id_cliente",
                (nome, telefone, email)
            )
            id_cliente = cursor.fetchone()[0]  # o id que o SERIAL acabou de gerar
            conexao.commit()
            return Cliente(nome, telefone, email, id_cliente)
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return None


def listar_clientes() -> list[Cliente]:
        clientes = []
        try:
            cursor.execute("SELECT id_cliente, nome, telefone, email FROM cliente ORDER BY id_cliente")
            for linha in cursor.fetchall():
                clientes.append(Cliente(linha[1], linha[2], linha[3], linha[0]))
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return clientes

def buscar_cliente(id_cliente: int) -> Cliente | None:
        try:
            cursor.execute(
                "SELECT id_cliente, nome, telefone, email FROM cliente WHERE id_cliente = %s",
                (id_cliente,)  # a vírgula faz disso uma tupla
            )
            linha = cursor.fetchone()  # None se nenhuma linha bateu com o WHERE
            if linha:
                return Cliente(linha[1], linha[2], linha[3], linha[0])
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return None

def atualizar_cliente(cliente: Cliente) -> bool:
        try:
            cursor.execute(
                "UPDATE cliente SET nome = %s, telefone = %s, email = %s WHERE id_cliente = %s",
                (cliente.nome, cliente.telefone, cliente.email, cliente.id_cliente)
            )
            alteradas = cursor.rowcount  # 0 = não existe cliente com esse id
            conexao.commit()
            return alteradas > 0
        except pg.DatabaseError as e:
            print(e)
            conexao.rollback()
        return False


clientes_para_inserir: list[Cliente] = [ 
    Cliente("Pedro Ricardo Souza", "51 93425644", "pedro.ricardo23@gmail.com"),
    Cliente("Pedro Augusto Silva", "47 98675632", "pedro.aug3412@gmail.com"),
    Cliente("Pedro Souza", "51 86655434", "pedro.souza2345@gmail.com"),
    Cliente("Ricardo José", "53 97985891", "ricardo.santos45@gmail.com"),
    Cliente("Fabricio Souza", None, None),
    Cliente("Joao Santos", None, None),
    Cliente("Joao Pedro Silva", None, None),]
print(clientes_para_inserir)

In [ ]:
# =====================================
# CLASSE QUADRA
# =====================================

class Quadra:

    def __init__(self, id_quadra: int, preco: float, coberta: bool):
        self.id_quadra = id_quadra
        self.preco = preco
        self.coberta = coberta

    def __str__(self):
        return (
            f"Quadra(id_quadra={self.id_quadra}, "
            f"preco=R$ {self.preco:.2f}, "
            f"coberta={self.coberta})"
        )

    def __repr__(self):
        return str(self)


# =====================================
# CRIAR QUADRA
# =====================================

def criar_quadra(preco: float, coberta: bool) -> Quadra | None:
    try:
        cursor.execute(
            """
            INSERT INTO quadra (preco, coberta)
            VALUES (%s, %s)
            RETURNING id_quadra
            """,
            (preco, coberta)
        )

        id_quadra = cursor.fetchone()[0]
        conexao.commit()

        return Quadra(id_quadra, preco, coberta)

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return None


# =====================================
# LISTAR QUADRAS
# =====================================

def listar_quadras() -> list[Quadra]:
    quadras = []

    try:
        cursor.execute(
            """
            SELECT id_quadra, preco, coberta
            FROM quadra
            ORDER BY id_quadra
            """
        )

        for linha in cursor.fetchall():
            quadras.append(
                Quadra(
                    linha[0],
                    linha[1],
                    linha[2]
                )
            )

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()

    return quadras


# =====================================
# BUSCAR QUADRA
# =====================================

def buscar_quadra(id_quadra: int) -> Quadra | None:
    try:
        cursor.execute(
            """
            SELECT id_quadra, preco, coberta
            FROM quadra
            WHERE id_quadra = %s
            """,
            (id_quadra,)
        )

        linha = cursor.fetchone()

        if linha:
            return Quadra(
                linha[0],
                linha[1],
                linha[2]
            )

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()

    return None


# =====================================
# ATUALIZAR QUADRA
# =====================================

def atualizar_quadra(quadra: Quadra) -> bool:
    try:
        cursor.execute(
            """
            UPDATE quadra
            SET preco = %s,
                coberta = %s
            WHERE id_quadra = %s
            """,
            (
                quadra.preco,
                quadra.coberta,
                quadra.id_quadra
            )
        )

        alteradas = cursor.rowcount
        conexao.commit()

        return alteradas > 0

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return False


# =====================================
# DELETAR QUADRA
# =====================================

def deletar_quadra(id_quadra: int) -> bool:
    try:
        cursor.execute(
            """
            DELETE FROM quadra
            WHERE id_quadra = %s
            """,
            (id_quadra,)
        )

        apagadas = cursor.rowcount
        conexao.commit()

        return apagadas > 0

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return False


# =====================================
# VALORES DAS QUADRAS PARA INSERIR
# =====================================

quadras = [
    (100.00, True),
    (100.00, True),
    (90.00, False),
    (90.00, False)
]


# =====================================
# INSERIR OS VALORES NO BANCO
# EXECUTAR APENAS UMA VEZ
# =====================================

# for preco, coberta in preco_das_quadras:
#     criar_quadra(preco, coberta)


# =====================================
# MOSTRAR TODAS AS QUADRAS
# =====================================

print("\n====== QUADRAS CADASTRADAS ======")

quadras = listar_quadras()

for quadra in quadras:
    print(quadra)


# =====================================
# EXEMPLOS DE UTILIZAÇÃO
# =====================================

# Buscar quadra pelo ID:
# resultado = buscar_quadra(1)
# print(resultado)


# Atualizar quadra:
# quadra = Quadra(1, 120.00, True)
# print(atualizar_quadra(quadra))


# Deletar quadra:
# print(deletar_quadra(1))


In [ ]:

import psycopg2 as pg


# =====================================
# CLASSE PARTIDA
# =====================================

class Partida:

    def __init__(self, id_partida: int, nome_artilheiro: str, total_gols: int):
        self.id_partida = id_partida
        self.nome_artilheiro = nome_artilheiro
        self.total_gols = total_gols

    def __str__(self):
        return (
            f"Partida(id_partida={self.id_partida}, "
            f"nome_artilheiro={self.nome_artilheiro}, "
            f"total_gols={self.total_gols})"
        )

    def __repr__(self):
        return str(self)


# =====================================
# CRIAR PARTIDA
# =====================================

def criar_partida(nome_artilheiro: str, total_gols: int) -> Partida | None:

    try:
        cursor.execute(
            """
            INSERT INTO PARTIDA (nome_artilheiro, total_gols)
            VALUES (%s, %s)
            RETURNING id_partida
            """,
            (nome_artilheiro, total_gols)
        )

        id_partida = cursor.fetchone()[0]
        conexao.commit()

        return Partida(id_partida, nome_artilheiro, total_gols)

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return None


# =====================================
# LISTAR PARTIDAS
# =====================================

def listar_partidas() -> list[Partida]:

    partidas = []

    try:
        cursor.execute(
            """
            SELECT id_partida, nome_artilheiro, total_gols
            FROM PARTIDA
            ORDER BY id_partida
            """
        )

        for linha in cursor.fetchall():
            partidas.append(
                Partida(linha[0], linha[1], linha[2])
            )

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()

    return partidas


# =====================================
# BUSCAR PARTIDA
# =====================================

def buscar_partida(id_partida: int) -> Partida | None:

    try:
        cursor.execute(
            """
            SELECT id_partida, nome_artilheiro, total_gols
            FROM PARTIDA
            WHERE id_partida = %s
            """,
            (id_partida,)
        )

        linha = cursor.fetchone()

        if linha:
            return Partida(linha[0], linha[1], linha[2])

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()

    return None


# =====================================
# ATUALIZAR PARTIDA
# =====================================

def atualizar_partida(partida: Partida) -> bool:

    try:
        cursor.execute(
            """
            UPDATE PARTIDA
            SET nome_artilheiro = %s,
                total_gols = %s
            WHERE id_partida = %s
            """,
            (
                partida.nome_artilheiro,
                partida.total_gols,
                partida.id_partida
            )
        )

        alteradas = cursor.rowcount
        conexao.commit()

        return alteradas > 0

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return False


# =====================================
# DELETAR PARTIDA
# =====================================

def deletar_partida(id_partida: int) -> bool:

    try:
        cursor.execute(
            """
            DELETE FROM PARTIDA
            WHERE id_partida = %s
            """,
            (id_partida,)
        )

        apagadas = cursor.rowcount
        conexao.commit()

        return apagadas > 0

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return False


# =====================================
# VALORES DAS PARTIDAS
# =====================================

partidas_para_inserir = [
    ("Carlos", 2),
    ("João", 3),
    ("Pedro", 1),
    ("Marcos", 4)
]


# =====================================
# INSERIR PARTIDAS NO POSTGRESQL
# =====================================

if __name__ == "__main__":

    print("\n====== INSERINDO PARTIDAS ======")

    for nome_artilheiro, total_gols in partidas_para_inserir:

        resultado = criar_partida(nome_artilheiro, total_gols)

        if resultado:
            print("Partida cadastrada:", resultado)

    # =====================================
    # MOSTRAR PARTIDAS CADASTRADAS
    # =====================================

    print("\n====== PARTIDAS CADASTRADAS ======")

    partidas = listar_partidas()

    for partida in partidas:
        print(partida)


    # =====================================
    # EXEMPLOS DE BUSCA, ATUALIZAÇÃO E EXCLUSÃO
    # =====================================

    # Buscar partida:
    # print(buscar_partida(1))

    # Atualizar partida:
    # partida = Partida(1, "Carlos", 5)
    # print(atualizar_partida(partida))

    # Deletar partida:
    # print(deletar_partida(1))


In [ ]:
class Agendamento:

    def __init__(
        self,
        id_agendamento: int,
        id_quadra: int,
        tempo_termino_max: int,
        hora: datetime,
        id_partida: int
    ):
        self.id_agendamento = id_agendamento
        self.id_quadra = id_quadra
        self.tempo_termino_max = tempo_termino_max
        self.hora = hora
        self.id_partida = id_partida

    def __str__(self):
        return (
            f"Agendamento(id={self.id_agendamento}, "
            f"quadra={self.id_quadra}, "
            f"tempo={self.tempo_termino_max}, "
            f"hora={self.hora}, "
            f"partida={self.id_partida})"
        )

    def __repr__(self):
        return str(self)


def criar_agendamento(
    id_agendamento: int,
    id_quadra: int,
    tempo_termino_max: int,
    hora: datetime,
    id_partida: int
) -> Agendamento | None:

    try:
        cursor.execute(
            """
            INSERT INTO AGENDAMENTO
            (id_agendamento, id_quadra, tempo_termino_max, hora, id_partida)
            VALUES (%s, %s, %s, %s, %s)
            RETURNING id_agendamento
            """,
            (
                id_agendamento,
                id_quadra,
                tempo_termino_max,
                hora,
                id_partida
            )
        )

        conexao.commit()

        return Agendamento(
            id_agendamento,
            id_quadra,
            tempo_termino_max,
            hora,
            id_partida
        )

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return None


def listar_agendamentos() -> list[Agendamento]:

    agendamentos = []

    try:
        cursor.execute(
            """
            SELECT id_agendamento, id_quadra,
                   tempo_termino_max, hora, id_partida
            FROM AGENDAMENTO
            ORDER BY id_agendamento
            """
        )

        for linha in cursor.fetchall():
            agendamentos.append(
                Agendamento(
                    linha[0],
                    linha[1],
                    linha[2],
                    linha[3],
                    linha[4]
                )
            )

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()

    return agendamentos


def buscar_agendamentos(id_agendamento: int) -> Agendamento | None:

    try:
        cursor.execute(
            """
            SELECT id_agendamento, id_quadra,
                   tempo_termino_max, hora, id_partida
            FROM AGENDAMENTO
            WHERE id_agendamento = %s
            """,
            (id_agendamento,)
        )

        linha = cursor.fetchone()

        if linha:
            return Agendamento(
                linha[0],
                linha[1],
                linha[2],
                linha[3],
                linha[4]
            )

        return None

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return None


def atualizar_agendamentos(agendamento: Agendamento) -> bool:

    try:
        cursor.execute(
            """
            UPDATE AGENDAMENTO
            SET id_quadra = %s,
                tempo_termino_max = %s,
                hora = %s,
                id_partida = %s
            WHERE id_agendamento = %s
            """,
            (
                agendamento.id_quadra,
                agendamento.tempo_termino_max,
                agendamento.hora,
                agendamento.id_partida,
                agendamento.id_agendamento
            )
        )

        alteradas = cursor.rowcount
        conexao.commit()

        return alteradas > 0

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return False


def deletar_agendamentos(id_agendamento: int) -> bool:

    try:
        cursor.execute(
            """
            DELETE FROM AGENDAMENTO
            WHERE id_agendamento = %s
            """,
            (id_agendamento,)
        )

        apagadas = cursor.rowcount
        conexao.commit()

        return apagadas > 0

    except pg.DatabaseError as e:
        print(e)
        conexao.rollback()
        return False


In [ ]:
partidas_para_inserir: list[Partida] = [  
    Partida("William", 12),      
    Partida("Bruno", 10),
    Partida("Jose S", 5),
    Partida("Gabriel Cunha", 11),
    Partida("Leonardo", 12),]
print(partidas_para_inserir)

[Partida William, Partida Bruno, Partida Jose S, Partida Gabriel Cunha, Partida Leonardo]


In [ ]:
preco_das_quadras: list[Quadra] = [
    Quadra(100,00, True),
    Quadra(100,00, True),
    Quadra(90,00, False), 
    Quadra(90,00, False),
    ]
print(preco_das_quadras)

[Quadra(100 - 0), Quadra(100 - 0), Quadra(90 - 0), Quadra(90 - 0)]


In [ ]:
agendamentos_para_inserir: list[Agendamento] = [
        (55, "20:00-2026-10-21"),
        (51, "20:00-2026-10-22"),
        (52, "20:00-2026-09-13"),
        (59, "20:00-2026-09-21"),
        (50, "20:00-2026-10-19"),
        (59, "20:00-2026-08-17"),
        (58, "20:00-2026-09-15"),
        (59, "20:00-2026-10-23"),]
print(agendamentos_para_inserir)

[(55, '20:00-2026-10-21'), (51, '20:00-2026-10-22'), (52, '20:00-2026-09-13'), (59, '20:00-2026-09-21'), (50, '20:00-2026-10-19'), (59, '20:00-2026-08-17'), (58, '20:00-2026-09-15'), (59, '20:00-2026-10-23')]
